# Cross-Process Simulator

Simulate the look of developing a film stock in the *wrong* chemistry using the parametric model in `tools/xpro_core.py`.

Pick a preset (or build a custom `XproProfile`), preview it, and export the parameters for the matching GLSL shader.


In [ ]:
# Make the tools/ package importable from the notebooks/ folder.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "tools")))
import numpy as np
import matplotlib.pyplot as plt
import xpro_core as xc
print("presets:", xc.list_presets())


In [ ]:
def sample_image(h=256, w=256):
    """A synthetic test image: gradients + color patches, no files needed."""
    y, x = np.mgrid[0:h, 0:w].astype(np.float32)
    r = x / (w - 1)
    g = y / (h - 1)
    b = (1 - r + g) / 2
    img = np.stack([r, g, b], -1)
    # drop in a few saturated patches and a gray ramp
    img[20:60, 20:60] = [0.9, 0.1, 0.1]
    img[20:60, 80:120] = [0.1, 0.7, 0.2]
    img[20:60, 140:180] = [0.2, 0.3, 0.9]
    img[h-50:h-20, :] = np.linspace(0, 1, w)[None, :, None]
    return np.clip(img, 0, 1)

img = sample_image()
plt.figure(figsize=(4,4)); plt.imshow(img); plt.title("synthetic input"); plt.axis("off");


## Apply a preset

Each preset corresponds to a shader in `shaders/`.

In [ ]:
preset = "c41_in_e6"  # try: e6_in_c41, push_xpro, expired_xpro, bw_in_c41
out = xc.apply_profile(img, xc.PRESETS[preset], seed=1)
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img); ax[0].set_title("original"); ax[0].axis("off")
ax[1].imshow(out); ax[1].set_title(preset); ax[1].axis("off");


## Compare all presets

In [ ]:
names = xc.list_presets()
fig, axes = plt.subplots(1, len(names), figsize=(3*len(names), 3))
for ax, name in zip(axes, names):
    ax.imshow(xc.apply_profile(img, xc.PRESETS[name], seed=2))
    ax.set_title(name, fontsize=9); ax.axis("off")
plt.tight_layout();


## Build a custom profile and export shader uniforms

The fields map directly onto the uniforms in the shaders (e.g. `u_contrast_boost`, `u_shift_intensity`, `u_saturation`).

In [ ]:
prof = xc.XproProfile(
    name="my_look",
    shadow_shift=(-0.03, 0.12, -0.05),
    midtone_shift=(0.14, -0.05, 0.11),
    highlight_shift=(0.16, 0.10, -0.14),
    contrast=7.0, pivot=0.45, saturation=1.4,
    grain_amount=0.06, grain_size=2.0, grain_shadow_bias=0.9,
)
plt.imshow(xc.apply_profile(img, prof, seed=3)); plt.axis("off"); plt.title(prof.name)
prof.to_json("my_look.json")
print("saved my_look.json — load it in any tool with --profile")
